<a href="https://colab.research.google.com/github/12thFulminata/llm-foundations/blob/main/00-how-llms-work/00_llm_foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
for text in ["I love pizza", "unbelievable", "strawberry", "ChatGPT is 🔥"]:
    ids = tok.encode(text)
    print(f"{text!r:18} -> {tok.convert_ids_to_tokens(ids)}  ids={ids}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

'I love pizza'     -> ['I', 'Ġlove', 'Ġpizza']  ids=[40, 1842, 14256]
'unbelievable'     -> ['un', 'bel', 'iev', 'able']  ids=[403, 6667, 11203, 540]
'strawberry'       -> ['st', 'raw', 'berry']  ids=[301, 1831, 8396]
'ChatGPT is 🔥'     -> ['Chat', 'G', 'PT', 'Ġis', 'ĠðŁ', 'Ķ', '¥']  ids=[30820, 38, 11571, 318, 12520, 242, 98]


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
prompt = "The capital of France is"
ids = tok(prompt, return_tensors="pt").input_ids

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

prompt = "The capital of France is"
out = tok(prompt, return_tensors="pt")

print(out)                # everything the tokenizer returned
print(out.input_ids)      # just the IDs
print(out.input_ids.shape)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

{'input_ids': tensor([[ 464, 3139,  286, 4881,  318]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}
tensor([[ 464, 3139,  286, 4881,  318]])
torch.Size([1, 5])


In [ ]:
tok.pad_token = tok.eos_token   # GPT-2 has no padding token, so reuse "end of text"

out = tok(["The capital of France is", "Hello"], return_tensors="pt", padding=True)
print(out.input_ids)
print(out.input_ids.shape)
print(out.attention_mask)

tensor([[  464,  3139,   286,  4881,   318],
        [15496, 50256, 50256, 50256, 50256]])
torch.Size([2, 5])
tensor([[1, 1, 1, 1, 1],
        [1, 0, 0, 0, 0]])


In [ ]:
for w in ["strawberry", " strawberry", " capital", " pizza", " tokenization", " Sufyan", " antidisestablishmentarianism"]:
    print(f"{w!r:35} -> {tok.convert_ids_to_tokens(tok.encode(w))}")

'strawberry'                        -> ['st', 'raw', 'berry']
' strawberry'                       -> ['Ġstrawberry']
' capital'                          -> ['Ġcapital']
' pizza'                            -> ['Ġpizza']
' tokenization'                     -> ['Ġtoken', 'ization']
' Sufyan'                           -> ['ĠSu', 'f', 'yan']
' antidisestablishmentarianism'     -> ['Ġant', 'idis', 'establishment', 'arian', 'ism']


In [ ]:
ids = tok("The capital of France is", return_tensors="pt").input_ids

with torch.no_grad():
    output = model(ids)

logits = output.logits
print(logits.shape)

torch.Size([1, 5, 50257])


In [ ]:
import torch

logits = torch.tensor([5.0, 3.0, 2.0])
probs = torch.softmax(logits, dim=-1)
print(probs)          # tensor([0.8438, 0.1142, 0.0420])
print(probs.sum())    # tensor(1.)

tensor([0.8438, 0.1142, 0.0420])
tensor(1.0000)


In [ ]:
ids = tok("The capital of France is", return_tensors="pt").input_ids
with torch.no_grad():
    logits = model(ids).logits

print(logits.shape)                        # should be [1, 5, 50257] again

last = logits[0, -1]
for word in [" Paris", " London", " pizza", " the"]:
    tid = tok.encode(word)[0]
    print(f"{word!r:10} id={tid:<6} score={last[tid]:.2f}")

torch.Size([1, 5, 50257])
' Paris'   id=6342   score=-101.21
' London'  id=3576   score=-103.94
' pizza'   id=14256  score=-111.92
' the'     id=262    score=-100.25


In [ ]:
p1 = torch.softmax(last, dim=-1)
p2 = torch.softmax(last + 10, dim=-1)

tid = tok.encode(" Paris")[0]
print(f"Paris before: {p1[tid]:.4%}")
print(f"Paris after:  {p2[tid]:.4%}")
print("All the same?", torch.allclose(p1, p2))

Paris before: 3.2245%
Paris after:  3.2245%
All the same? True


In [ ]:
ids = tok("The capital of France is", return_tensors="pt").input_ids

for step in range(15):
    with torch.no_grad():
        logits = model(ids).logits
    next_id = logits[0, -1].argmax()
    ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
    print(f"step {step+1:2} | shape {list(ids.shape)} | {tok.decode(ids[0])}")

step  1 | shape [1, 6] | The capital of France is the
step  2 | shape [1, 7] | The capital of France is the capital
step  3 | shape [1, 8] | The capital of France is the capital of
step  4 | shape [1, 9] | The capital of France is the capital of the
step  5 | shape [1, 10] | The capital of France is the capital of the French
step  6 | shape [1, 11] | The capital of France is the capital of the French Republic
step  7 | shape [1, 12] | The capital of France is the capital of the French Republic,
step  8 | shape [1, 13] | The capital of France is the capital of the French Republic, and
step  9 | shape [1, 14] | The capital of France is the capital of the French Republic, and the
step 10 | shape [1, 15] | The capital of France is the capital of the French Republic, and the capital
step 11 | shape [1, 16] | The capital of France is the capital of the French Republic, and the capital of
step 12 | shape [1, 17] | The capital of France is the capital of the French Republic, and the capital of

In [ ]:
out = model.generate(tok("The capital of France is", return_tensors="pt").input_ids,
                     max_new_tokens=15, do_sample=False, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0]))

The capital of France is the capital of the French Republic, and the capital of the French Republic is


In [ ]:
prompt = "The capital of France is"
ids = tok(prompt, return_tensors="pt").input_ids
#torch.manual_seed(42)

settings = [
    ("greedy",       dict(do_sample=False)),
    ("temp 0.3",     dict(do_sample=True, temperature=0.3)),
    ("temp 1.0",     dict(do_sample=True, temperature=1.0)),
    ("temp 2.0",     dict(do_sample=True, temperature=2.0)),
    ("temp 0.8 + top_p 0.9", dict(do_sample=True, temperature=0.8, top_p=0.9)),
    ("greedy + repetition penalty", dict(do_sample=False, repetition_penalty=1.3)),
]

for name, kwargs in settings:
    out = model.generate(ids, max_new_tokens=30, pad_token_id=tok.eos_token_id, **kwargs)
    print(f"--- {name} ---\n{tok.decode(out[0])}\n")

--- greedy ---
The capital of France is the capital of the French Republic, and the capital of the French Republic is the capital of the French Republic.

The French Republic is the capital

--- temp 0.3 ---
The capital of France is the capital of France, and the capital of France is the capital of France.

The capital of France is the capital of France.



--- temp 1.0 ---
The capital of France is a former bastion of Catholic values for the middle classes — and this is hardly the sort of land they now want to see return to.



--- temp 2.0 ---
The capital of France is not a city where the population would do something with itself.[16][9]

While the state-made air space was necessary due to military

--- temp 0.8 + top_p 0.9 ---
The capital of France is home to the biggest mosque in the world, the Al-Azhar mosque in the heart of Paris. The mosque, which is also home to the

--- greedy + repetition penalty ---
The capital of France is the city, and its inhabitants are not only rich but 

In [ ]:
type(settings)

list

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager")

sentence = "The animal didn't cross the street because it was too tired"
inp = tok(sentence, return_tensors="pt")

with torch.no_grad():
    out = model(**inp, output_attentions=True)

print("layers:", len(out.attentions))
print("shape of one layer:", out.attentions[0].shape)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

layers: 12
shape of one layer: torch.Size([1, 12, 12, 12])


In [ ]:
a_pos = tokens.index("Ġanimal")
s_pos = tokens.index("Ġstreet")

for layer in range(12):
    att = out.attentions[layer][0][:, it_pos]     # [heads, tokens]: each head's row for "it"
    h = att[:, a_pos].argmax()                    # the head that looks at "animal" the most
    print(f"layer {layer:2} head {h.item():2} | animal {att[h, a_pos]:.0%} | "
          f"street {att[h, s_pos]:.0%} | sink {att[h, 0]:.0%}")

layer  0 head  6 | animal 21% | street 23% | sink 11%
layer  1 head  6 | animal 28% | street 2% | sink 47%
layer  2 head  1 | animal 9% | street 4% | sink 78%
layer  3 head  5 | animal 29% | street 3% | sink 44%
layer  4 head  3 | animal 85% | street 0% | sink 8%
layer  5 head 10 | animal 41% | street 2% | sink 10%
layer  6 head  0 | animal 40% | street 1% | sink 38%
layer  7 head  8 | animal 15% | street 10% | sink 31%
layer  8 head  5 | animal 15% | street 4% | sink 35%
layer  9 head  0 | animal 36% | street 4% | sink 37%
layer 10 head  0 | animal 12% | street 5% | sink 75%
layer 11 head  0 | animal 20% | street 7% | sink 17%


In [ ]:
sentence = "The animal didn't cross the street because it was too wide"
inp = tok(sentence, return_tensors="pt")
with torch.no_grad():
    out = model(**inp, output_attentions=True)
tokens = tok.convert_ids_to_tokens(inp.input_ids[0])
it_pos = tokens.index("Ġit")

In [ ]:
a_pos = tokens.index("Ġanimal")
s_pos = tokens.index("Ġstreet")

for layer in range(12):
    att = out.attentions[layer][0][:, it_pos]     # [heads, tokens]: each head's row for "it"
    h = att[:, a_pos].argmax()                    # the head that looks at "animal" the most
    print(f"layer {layer:2} head {h.item():2} | animal {att[h, a_pos]:.0%} | "
          f"street {att[h, s_pos]:.0%} | sink {att[h, 0]:.0%}")

layer  0 head  6 | animal 21% | street 23% | sink 11%
layer  1 head  6 | animal 28% | street 2% | sink 47%
layer  2 head  1 | animal 9% | street 4% | sink 78%
layer  3 head  5 | animal 29% | street 3% | sink 44%
layer  4 head  3 | animal 85% | street 0% | sink 8%
layer  5 head 10 | animal 41% | street 2% | sink 10%
layer  6 head  0 | animal 40% | street 1% | sink 38%
layer  7 head  8 | animal 15% | street 10% | sink 31%
layer  8 head  5 | animal 15% | street 4% | sink 35%
layer  9 head  0 | animal 36% | street 4% | sink 37%
layer 10 head  0 | animal 12% | street 5% | sink 75%
layer 11 head  0 | animal 20% | street 7% | sink 17%


In [ ]:
for word in ["tired", "wide"]:
    s = f"The animal didn't cross the street because it was too {word}"
    inp = tok(s, return_tensors="pt")
    with torch.no_grad():
        out = model(**inp, output_attentions=True)
    tokens = tok.convert_ids_to_tokens(inp.input_ids[0])
    last = len(tokens) - 1                                  # position of "tired"/"wide"
    a_pos, s_pos = tokens.index("Ġanimal"), tokens.index("Ġstreet")

    print(f"\n--- '{word}' looking back ---")
    for layer in range(12):
        att = out.attentions[layer][0][:, last]              # [heads, tokens]
        a, s_ = att[:, a_pos].max(), att[:, s_pos].max()     # strongest head for each
        print(f"layer {layer:2} | best head on animal {a:.0%} | best head on street {s_:.0%}")


--- 'tired' looking back ---
layer  0 | best head on animal 16% | best head on street 21%
layer  1 | best head on animal 9% | best head on street 11%
layer  2 | best head on animal 5% | best head on street 10%
layer  3 | best head on animal 14% | best head on street 14%
layer  4 | best head on animal 34% | best head on street 6%
layer  5 | best head on animal 34% | best head on street 4%
layer  6 | best head on animal 5% | best head on street 18%
layer  7 | best head on animal 2% | best head on street 3%
layer  8 | best head on animal 8% | best head on street 8%
layer  9 | best head on animal 10% | best head on street 5%
layer 10 | best head on animal 6% | best head on street 11%
layer 11 | best head on animal 14% | best head on street 83%

--- 'wide' looking back ---
layer  0 | best head on animal 23% | best head on street 21%
layer  1 | best head on animal 9% | best head on street 9%
layer  2 | best head on animal 10% | best head on street 20%
layer  3 | best head on animal 12% | be

In [ ]:
print(model.config.n_positions)                # max positions
print(model.transformer.wpe.weight.shape)      # the position table: [1024, 768]

1024
torch.Size([1024, 768])


In [ ]:
long_ids = torch.randint(0, 50257, (1, 1100))   # 1,100 random token IDs

try:
    with torch.no_grad():
        model(long_ids)
    print("worked?!")
except Exception as e:
    print("Error:", type(e).__name__, "-", e)

Error: IndexError - index out of range in self


In [ ]:
for n in [1_024, 8_000, 128_000, 1_000_000]:
    print(f"{n:>9,} tokens → {n*n:>22,} attention scores per head, per layer")

    1,024 tokens →              1,048,576 attention scores per head, per layer
    8,000 tokens →             64,000,000 attention scores per head, per layer
  128,000 tokens →         16,384,000,000 attention scores per head, per layer
1,000,000 tokens →      1,000,000,000,000 attention scores per head, per layer


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

q = "What is the capital of France?"

# 1) Base model: pretraining only
base_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B")
ids = base_tok(q, return_tensors="pt").input_ids
out = base.generate(ids, max_new_tokens=40, do_sample=False)
print("=== BASE ===\n" + base_tok.decode(out[0]))

# 2) Instruct model: pretraining + SFT + preference tuning
chat_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
chat = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": q},
]
text = chat_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("=== WHAT THE MODEL ACTUALLY SEES ===\n" + text)

ids = chat_tok(text, return_tensors="pt").input_ids
out = chat.generate(ids, max_new_tokens=40, do_sample=False)
answer = chat_tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
print("=== INSTRUCT ===\n" + answer)

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

... (10 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

=== WHAT THE MODEL ACTUALLY SEES ===
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant

=== INSTRUCT ===
The capital of France is Paris.


In [ ]:
ids = base_tok(q, return_tensors="pt").input_ids
out = base.generate(ids, max_new_tokens=40, do_sample=False)
print("=== BASE ===\n" + base_tok.decode(out[0]))

=== BASE ===
What is the capital of France? The capital of France is Paris.<|endoftext|>


In [ ]:
task = "Give me exactly 3 tips for learning Python, as a bulleted list. No intro."

# base
ids = base_tok(task, return_tensors="pt").input_ids
out = base.generate(ids, max_new_tokens=80, do_sample=False)
print("=== BASE ===\n" + base_tok.decode(out[0]))

# instruct
msgs = [{"role": "user", "content": task}]
text = chat_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
ids = chat_tok(text, return_tensors="pt").input_ids
out = chat.generate(ids, max_new_tokens=80, do_sample=False)
print("=== INSTRUCT ===\n" + chat_tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

=== BASE ===
Give me exactly 3 tips for learning Python, as a bulleted list. No intro.<|endoftext|>
=== INSTRUCT ===
1. **Practice Regularly**: The best way to learn any programming language is through practice. Try to solve problems and write code regularly.
2. **Study the Basics**: Start with the basics of Python, including data types, control structures (if-else statements), functions, and basic syntax. Understanding these foundational concepts will help you build more complex programs later on.
3. **Join a Community


In [ ]:
fake = "Summarize the 2021 paper 'Quantum Tokenizers for Llamas' by Sufyan Shah in 3 sentences."
msgs = [{"role": "user", "content": fake}]
text = chat_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
ids = chat_tok(text, return_tensors="pt").input_ids
out = chat.generate(ids, max_new_tokens=120, do_sample=False)
print(chat_tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

The 2021 paper "Quantum Tokenizers for Llamas" by Sufyan Shah introduces a novel approach to quantum computing that could revolutionize machine learning and natural language processing tasks. The authors propose using quantum computers to tokenize text data, which is crucial for tasks such as sentiment analysis, named entity recognition, and more. By leveraging the power of quantum mechanics, they aim to achieve significant improvements in computational efficiency and accuracy compared to classical approaches. This work represents a significant advancement in the field of quantum computing and has the potential to transform various applications, including but not limited to,


In [ ]:
new_tokens = out[0][ids.shape[1]:]
print(len(new_tokens))                    # 120 = hit the limit
print(chat_tok.decode(new_tokens[-1:]))   # the very last token

120
,


In [ ]:
text = chat_tok.decode(new_tokens)
print("words: ", len(text.split()))
print("tokens:", len(new_tokens))
print(chat_tok.convert_ids_to_tokens(new_tokens)[:40])   # first 40 tokens

words:  98
tokens: 120
['The', 'Ġ', '2', '0', '2', '1', 'Ġpaper', 'Ġ"', 'Quant', 'um', 'ĠToken', 'izers', 'Ġfor', 'ĠL', 'lam', 'as', '"', 'Ġby', 'ĠS', 'uf', 'yan', 'ĠShah', 'Ġintroduces', 'Ġa', 'Ġnovel', 'Ġapproach', 'Ġto', 'Ġquantum', 'Ġcomputing', 'Ġthat', 'Ġcould', 'Ġrevolution', 'ize', 'Ġmachine', 'Ġlearning', 'Ġand', 'Ġnatural', 'Ġlanguage', 'Ġprocessing', 'Ġtasks']
